# VISSIM 원시 자료 → 10초 시계열 CSV

VISSIM 검지기 원시 자료(parquet)를 10초 단위로 집계하고 10회(N회) 평균을 내서 `03_TimeSeries_Sample` 양식의 CSV로 저장합니다.

| 순서 | 셀 | 하는 일 |
|---|---|---|
| 0 | **설정** | 경로, 집계 시간, 표기 규칙 등 자주 고치는 값 — **여기만 고치면 됩니다** |
| 1 | 1단계 | 원시 자료 확인 (검지기 목록, 시나리오–파일 짝, 실행 번호, 컬럼) |
| 2 | 2단계 | 시드(실행 1개)별 10초 집계 (교통량, 속도, 점유율, 정지 여부) |
| 3 | 3단계 | 10회 평균, 반올림, NA 표기, CSV 저장 |
| 4 | 실행 | 처리할 구간·시나리오를 고르고 실행 (진행률 표시) |
| 5 | 결과 확인 | 실행 기록 요약, 결과 파일 미리 보기 |

**위에서부터 차례로 실행**하세요 (메뉴: Run All 또는 셀마다 Shift+Enter).

필요한 패키지: `pandas`, `numpy`, `pyarrow`, `openpyxl` / 병렬 처리를 빠르게 하려면 `joblib` (없으면 스레드로 처리 — 더 느림)

**결과 속도 컬럼(speed_kmh)의 뜻**: 숫자 = 주행 · `0.0` = 정체(차가 10초 내내 검지기 위에 정지) · `NA` = 새로 진입한 차량 없음

## 0. 설정 — 자주 바뀌는 값

In [ ]:
# ---------------------------------------------------------------- 경로
# VISSIM 원시 자료(parquet) — 세트별 최상위 폴더. 그 아래에 구간 폴더(SECTIONS의 raw_dir)가 있어야 합니다.
RAW_DIRS = {
    "추정용": r"D:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-추정용,검증용)_261022\1. 추정용 시나리오",
    "검증용": r"D:\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\VISSIM_Workspace\시나리오(140-추정용,검증용)_261022\2. 검증용 시나리오",
}
# 유고 시나리오 표(CSV) — 세트별 폴더와 파일 이름 형식 ({name} = SECTIONS의 scenario_csv)
SCENARIO_CSV_DIRS = {
    "추정용": r"\\naeilenc\도로교통안전연구소\4. 함민진\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\4. 유고 시나리오\1. 추정용 시나리오",
    "검증용": r"\\naeilenc\도로교통안전연구소\4. 함민진\2026년\1. 초장대 K-지하고속도로 인프라 안전 및 효율 향상 기술 개발\4. 유고 시나리오\2. 검증용 시나리오",
}
SCENARIO_CSV_NAME = "synthetic_scenarios_all_140_with_set_type({name}).csv"
SCENARIO_CSV_ENCODING = "cp949"
# 검지기 목록 — 양식 파일의 02_Detector_Info 시트 (결과 행 순서도 이 시트 순서)
DETECTOR_XLSX = r"C:\Users\alswl\Desktop\회복모형 분석\회복모형 분석자료\회복모형 시뮬레이션 분석자료_261111\1. 회복모형 데이터 요청 양식\VISSIM_회복모형_데이터요청_양식_수정(진입부).xlsx"
DETECTOR_SHEET = "02_Detector_Info"
# 결과 CSV를 저장할 폴더 (구간별 하위 폴더가 자동으로 만들어짐)
OUTPUT_DIR = r"C:\Users\alswl\Desktop\회복모형 분석\시계열_출력"

# ---------------------------------------------------------------- 시간
RAW_START_SEC = 1800     # 원시 기록 시작 (이보다 앞선 진입은 집계하지 않음)
RAW_END_SEC = 13200      # 원시 기록 끝 (끝까지 검지기 위에 남은 차는 이 시각에 벗어난 것으로 봄)
BIN_SEC = 10             # 집계 시간 (초) — 바꾸면 위아래 시각들이 이 값의 배수인지 확인
OUT_START_SEC = 1800     # 결과 파일 첫 칸의 시작 시각
OUT_END_SEC = 12600      # 결과 파일 마지막 칸의 끝 시각 (마지막 칸 = 12,590~12,600초)

# ---------------------------------------------------------------- 계산·표기 규칙
STOPPED_SEED_AS_ZERO = True      # 정체로 10초 내내 멈춘 시드를 속도 0으로 평균에 포함 (False = 예전 방식: 제외)
NO_MEASURE_SPEED = "NA"          # 모든 시드가 제외된 칸의 속도 표기: "NA" / "" (빈칸) / 0 (예전 방식)
SPEED_DECIMALS = 2               # 속도 소수 자리 (numpy 반올림 — 기존 결과와 같게 유지)
OCCUPANCY_DECIMALS = 1           # 점유율 소수 자리 (사사오입)
VOLUME_MIN_ONE_WITH_SPEED = True # 교통량 평균이 반올림으로 0이 되어도 속도가 있으면 1
RUN_TYPE = "base"
NOTES = "{runs}회 평균"          # {runs} = 시나리오당 실행 횟수
OUTPUT_ENCODING = "utf-8-sig"    # 엑셀에서 한글이 깨지지 않도록
OUTPUT_FILE_NAME = "Scenario_{sid:03d}_{runs}회평균_{suffix}.csv"

# ---------------------------------------------------------------- 원시 자료(parquet) 컬럼 이름
COL_DETECTOR, COL_ENTRY, COL_EXIT, COL_VEHICLE, COL_SPEED = "Measurem.", "t(Entry)", "t(Exit)", "VehNo", "v[km/h]"

# ---------------------------------------------------------------- 구간 목록
# key: 실행할 때 고르는 이름 / raw_dir: RAW_DIRS 아래 폴더 / scenario_csv: 시나리오 표 이름 / out_dir: 결과 폴더 / suffix: 파일 이름 끝
SECTIONS = [
    dict(key="진입구간", raw_dir=r"진입구간", scenario_csv="진입구간", out_dir=r"진입구간", suffix="진입부"),
    *[dict(key=f"본선부-구간{i}", raw_dir=rf"본선부\구간{i}", scenario_csv=f"본선부-구간{i}",
           out_dir=rf"본선부\구간{i}", suffix=f"본선부_구간{i}") for i in range(1, 6)],
    *[dict(key=f"영향권{i}({k})", raw_dir=rf"연결로 영향권\영향권{i}", scenario_csv=f"연결로-영향권{i}",
           out_dir=rf"연결로 영향권\영향권{i}", suffix=f"연결로 영향권{i}_{k}")
      for i, k in zip(range(1, 6), ["진출", "진입", "진출", "진입", "진출"])],
    dict(key="진출구간", raw_dir=r"진출구간", scenario_csv="진출구간", out_dir=r"진출구간", suffix="진출부"),
]
SETS = ["추정용", "검증용"]

# ---------------------------------------------------------------- 실행
WORKERS = 16          # 동시에 처리할 시나리오 수
OVERWRITE = False     # True면 이미 있는 결과 파일도 다시 만듦

라이브러리를 불러옵니다.

In [ ]:
import os, re, glob, time, datetime
import numpy as np, pandas as pd, pyarrow.parquet as pq, openpyxl
from concurrent.futures import ThreadPoolExecutor, as_completed
try:
    from joblib import Parallel, delayed          # 있으면 프로세스 병렬 (빠름)
    HAS_JOBLIB = True
except ImportError:
    HAS_JOBLIB = False
print("병렬 처리:", "joblib 프로세스" if HAS_JOBLIB else "스레드 (joblib 없음 — pip install joblib 하면 더 빠름)")

## 1단계: 원시 자료 확인
- 검지기 목록(양식 02 시트)을 읽습니다.
- 구간·세트마다 시나리오 표와 parquet 파일의 짝을 맞춥니다.
  - 파일은 이름 끝 실행 번호(`_1 … _N`) 순서로 정렬합니다. 이름순으로 정렬하면 `_1000`이 `_100` 뒤에 와서 짝이 어긋납니다.
  - 실행 번호가 1..N으로 빠짐없는지, 파일 수가 시나리오 수의 배수인지 확인합니다.
  - 시나리오당 실행 횟수 = 파일 수 ÷ 시나리오 수입니다. 두 세트의 실행 횟수가 같아야 합니다.
- 구간마다 첫 파일을 열어 필요한 컬럼과 검지기 번호가 있는지 확인합니다.
- 문제가 있는 구간·세트는 작업 목록에서 빼고 이유를 보여 줍니다.

In [ ]:
def load_detectors():
    """02_Detector_Info 시트의 검지기(D로 시작) → detector_location_id, detector_id, lane_id"""
    ws = openpyxl.load_workbook(DETECTOR_XLSX, read_only=True)[DETECTOR_SHEET]
    head = next(ws.iter_rows(max_row=1, values_only=True))
    rows = [r for r in ws.iter_rows(min_row=2, values_only=True) if r[1] and str(r[1]).startswith("D")]
    d = pd.DataFrame(rows, columns=head)[["detector_location_id", "detector_id", "lane_id"]].reset_index(drop=True)
    if not d.detector_id.is_unique:
        raise ValueError("02 시트에 같은 detector_id가 두 번 이상 있습니다")
    return d


def detector_index(det):
    """VISSIM 검지기 번호 → 결과 행 순서 (예: 'D10001' → 10001)"""
    return {int(x[1:]): i for i, x in enumerate(det.detector_id)}


def scenario_table(set_type, section):
    path = os.path.join(SCENARIO_CSV_DIRS[set_type], SCENARIO_CSV_NAME.format(name=section["scenario_csv"]))
    if not os.path.exists(path):
        raise FileNotFoundError(f"시나리오 표 없음: {path}")
    return pd.read_csv(path, encoding=SCENARIO_CSV_ENCODING).sort_values("scenario_id").reset_index(drop=True)


def run_number(path):
    m = re.search(r"_(\d+)\.parquet$", os.path.basename(path))
    if not m:
        raise ValueError(f"파일 이름 끝에 실행 번호가 없음: {path}")
    return int(m.group(1))


def run_files(set_type, section):
    """→ (시나리오 표, {scenario_id: [실행 파일 N개]}, N)"""
    folder = os.path.join(RAW_DIRS[set_type], section["raw_dir"])
    files = glob.glob(os.path.join(folder, "*.parquet"))
    if not files:
        raise FileNotFoundError(f"parquet 없음: {folder}")
    files = sorted(files, key=run_number)
    nums = [run_number(f) for f in files]
    if nums != list(range(1, len(files) + 1)):
        miss = sorted(set(range(1, len(files) + 1)) - set(nums))[:10]
        raise ValueError(f"실행 번호가 1..{len(files)}이 아님 (빠진 번호 예: {miss}) — {folder}")
    sc = scenario_table(set_type, section)
    if len(files) % len(sc):
        raise ValueError(f"파일 {len(files)}개가 시나리오 {len(sc)}개의 배수가 아님 (실행 중이거나 빠진 파일) — {folder}")
    n = len(files) // len(sc)
    return sc, {int(s): files[n * j:n * j + n] for j, s in enumerate(sc.scenario_id)}, n


def check_first_file(path, det_idx):
    need = [COL_DETECTOR, COL_ENTRY, COL_EXIT, COL_VEHICLE, COL_SPEED]
    miss_cols = [c for c in need if c not in pq.read_schema(path).names]
    if miss_cols:
        return f"컬럼 없음 {miss_cols}"
    raw = set(pq.read_table(path, columns=[COL_DETECTOR]).column(0).to_pylist())
    miss = sorted(set(det_idx) - raw)
    return f"원시 자료에 없는 검지기 {len(miss)}개 (예: {miss[:5]})" if miss else ""


def build_tasks(sections=None, sets=None, scenario_ids=None, verbose=True):
    """→ (작업 목록, 검지기 표, 문제 목록). 작업 1개 = (구간 dict, 세트, scenario_id, [실행 파일], 실행 횟수)"""
    det = load_detectors(); idx = detector_index(det)
    unknown = set(sections or []) - {s["key"] for s in SECTIONS}
    if unknown:
        raise ValueError(f"SECTIONS에 없는 구간: {sorted(unknown)} (가능: {[s['key'] for s in SECTIONS]})")
    secs = [s for s in SECTIONS if not sections or s["key"] in sections]
    tasks, problems, rows = [], [], []
    for sec in secs:
        sec_tasks, runs = [], set()
        for st in (sets or SETS):
            try:
                sc, files, n = run_files(st, sec)
                msg = check_first_file(files[int(sc.scenario_id.iloc[0])][0], idx)
                if msg:
                    raise ValueError(msg)
            except Exception as e:
                problems.append((sec["key"], st, str(e))); rows.append((sec["key"], st, "-", "-", "문제: " + str(e)[:80])); continue
            runs.add(n)
            ids = [s for s in files if not scenario_ids or s in scenario_ids]
            sec_tasks += [(sec, st, s, files[s], n) for s in ids]
            rows.append((sec["key"], st, len(files), n, f"OK ({len(ids)}개 처리)"))
        if len(runs) > 1:
            problems.append((sec["key"], "-", f"세트끼리 실행 횟수가 다름 {sorted(runs)}")); continue
        tasks += sec_tasks
    if verbose:
        print(f"검지기 {len(det)}개 (양식 02 시트)")
        print(pd.DataFrame(rows, columns=["구간", "세트", "시나리오 수", "실행 횟수", "상태"]).to_string(index=False))
        print(f"→ 처리할 시나리오 {len(tasks)}개" + (f", 문제 {len(problems)}건 (해당 구간·세트는 건너뜀)" if problems else ""))
        for p in problems:
            print("  문제:", " | ".join(p))
    return tasks, det, problems

확인만 해 보려면 아래 셀을 실행합니다. 원시 자료를 읽기만 하고, 1~2분 걸립니다.

In [ ]:
_ = build_tasks()

## 2단계: 시드(실행 1개)별 10초 집계
원시 기록에서는 차 1대가 검지기를 지날 때 2행이 생깁니다.
- 진입 행: `t(Exit) = -1`, 진입 속도가 들어 있음
- 진출 행: `t(Entry) = -1`

| 값 | 계산 |
|---|---|
| 교통량 | 그 칸에 진입한 차 수 |
| 속도 | 그 칸에 진입한 차들의 진입 속도 평균 (진입한 차가 없으면 비움) |
| 점유율 | 차가 검지기 위에 있던 시간(진입~진출) 중 그 칸에 걸친 부분의 합 ÷ 칸 길이 × 100. 경계에 걸친 차는 양쪽 칸에 나눠 넣음 |
| 정지 | 그 칸에 진입한 차는 없고, 칸 시작 이전에 진입해 칸 끝 이후에 벗어난 차가 있음 (10초 내내 같은 차가 멈춰 있음) |

기록이 잘린 차도 처리합니다.
- 진출 기록만 있으면 `RAW_START_SEC`에 진입한 것으로 봅니다.
- 진입 기록만 있으면 `RAW_END_SEC`에 벗어난 것으로 봅니다.

In [ ]:
def n_bins():
    return (RAW_END_SEC - RAW_START_SEC) // BIN_SEC


def seed_arrays(path, det_idx):
    """→ dict(volume, speed, occupancy, stopped) — 각각 (검지기 수 × 칸 수) 배열"""
    T0, T1, B = RAW_START_SEC, RAW_END_SEC, BIN_SEC
    NB, nd = n_bins(), len(det_idx)
    t = pq.read_table(path, columns=[COL_DETECTOR, COL_ENTRY, COL_EXIT, COL_VEHICLE, COL_SPEED]).to_pandas()
    t = t[t[COL_DETECTOR].isin(det_idx)]
    ent, ext = t[t[COL_ENTRY] >= 0], t[t[COL_EXIT] >= 0]

    # 교통량·속도: 진입 시각이 속한 칸
    e = ent[(ent[COL_ENTRY] >= T0) & (ent[COL_ENTRY] < T1)]
    di = e[COL_DETECTOR].map(det_idx).values
    bi = ((e[COL_ENTRY].values - T0) // B).astype(int)
    vol = np.bincount(di * NB + bi, minlength=nd * NB).reshape(nd, NB).astype(float)
    ssum = np.bincount(di * NB + bi, weights=e[COL_SPEED].values, minlength=nd * NB).reshape(nd, NB)
    with np.errstate(invalid="ignore", divide="ignore"):
        spd = np.where(vol > 0, ssum / vol, np.nan)

    # 같은 차의 진입 행과 진출 행을 짝지음 (잘린 기록은 시작·끝 시각으로 채움)
    p = ent[[COL_DETECTOR, COL_VEHICLE, COL_ENTRY]].merge(
        ext[[COL_DETECTOR, COL_VEHICLE, COL_EXIT]], on=[COL_DETECTOR, COL_VEHICLE], how="outer")
    p[COL_ENTRY] = p[COL_ENTRY].fillna(T0); p[COL_EXIT] = p[COL_EXIT].fillna(T1)

    # 점유율: 검지기 위에 있던 시간을 칸별로 나눠 더함
    q = p[(p[COL_EXIT] > T0) & (p[COL_ENTRY] < T1) & (p[COL_EXIT] >= p[COL_ENTRY])]
    a = np.clip(q[COL_ENTRY].values, T0, T1); b = np.clip(q[COL_EXIT].values, T0, T1)
    d = q[COL_DETECTOR].map(det_idx).values
    occ = np.zeros((nd, NB))
    b0 = ((a - T0) // B).astype(int); b1 = np.minimum(((b - T0) // B).astype(int), NB - 1)
    same = b0 == b1
    np.add.at(occ, (d[same], b0[same]), (b - a)[same])
    for k in np.where(~same)[0]:                        # 칸 경계에 걸친 차만 나눠 넣음
        for j in range(b0[k], b1[k] + 1):
            lo, hi = max(a[k], T0 + j * B), min(b[k], T0 + (j + 1) * B)
            if hi > lo:
                occ[d[k], j] += hi - lo
    occ = occ / B * 100

    # 정지: 칸 시작 이전 진입 ~ 칸 끝 이후 진출인 차가 있고, 그 칸에 진입한 차는 없음
    first = np.maximum(np.ceil((p[COL_ENTRY].values - T0) / B).astype(int), 0)            # 진입 이후 시작하는 첫 칸
    last = np.minimum(np.floor((p[COL_EXIT].values - T0) / B).astype(int) - 1, NB - 1)  # 진출 전에 끝나는 마지막 칸
    pd_ = p[COL_DETECTOR].map(det_idx).values
    ok = last >= first
    diff = np.zeros((nd, NB + 1), dtype=np.int64)       # 구간 [first, last]에 +1 (차이 배열 → 누적합)
    np.add.at(diff, (pd_[ok], first[ok]), 1)
    np.add.at(diff, (pd_[ok], last[ok] + 1), -1)
    cover = np.cumsum(diff, axis=1)[:, :NB] > 0
    stopped = cover & (vol == 0)

    return dict(volume=vol, speed=spd, occupancy=occ, stopped=stopped)

## 3단계: 10회(N회) 평균, 표기, CSV 저장

**속도** — 시드마다 아래처럼 정한 뒤, 남은 값으로 평균을 냅니다.

| 시드 상황 | 시드 값 | 평균에 |
|---|---|---|
| ① 그 칸에 진입한 차가 있음 | 진입 속도 평균 | 포함 |
| ② 진입한 차는 없고, 차가 칸 전체 동안 멈춰 있음 | 0 | 포함 (`STOPPED_SEED_AS_ZERO`) |
| ③ 그 밖 (차 없음, 앞서 진입한 차가 벗어나기만 함) | — | 제외 |

모든 시드가 제외되면 `NO_MEASURE_SPEED`(기본 `NA`)로 둡니다.

**교통량**: N회 단순 평균 → 사사오입 정수. 속도가 있으면 최소 1입니다.

**점유율**: N회 단순 평균 → 사사오입 소수 1자리.

**행** = 검지기(02 시트 순서) × 칸(`OUT_START_SEC` ~ `OUT_END_SEC`)

**컬럼** = 양식 `03_TimeSeries_Sample`의 11개

In [ ]:
COLS = ["scenario_id", "time_sec", "detector_location_id", "detector_id", "lane_id", "speed_kmh", "volume",
        "occupancy_pct", "aggregation_sec", "run_type", "notes"]


def half_up(x, d=0):
    """사사오입 (엑셀 ROUND와 같음: 4.5 → 5, 34.35 → 34.4). 1e-9는 이진수 오차 보정."""
    return np.floor(np.asarray(x, dtype=float) * 10 ** d + 0.5 + 1e-9) / 10 ** d


def average_seeds(seeds):
    if STOPPED_SEED_AS_ZERO:
        S = np.stack([np.where(s["stopped"], 0.0, s["speed"]) for s in seeds])
    else:
        S = np.stack([s["speed"] for s in seeds])
    with np.errstate(invalid="ignore"):
        spd = np.nanmean(S, axis=0)                     # 제외된 시드(NaN)는 빼고 평균, 모두 제외면 NaN
    vol = np.mean(np.stack([s["volume"] for s in seeds]), axis=0)
    occ = np.mean(np.stack([s["occupancy"] for s in seeds]), axis=0)
    return spd, vol, occ


def output_bins():
    s0 = (OUT_START_SEC - RAW_START_SEC) // BIN_SEC
    s1 = (OUT_END_SEC - RAW_START_SEC) // BIN_SEC
    if not (0 <= s0 < s1 <= n_bins()):
        raise ValueError("OUT_START_SEC·OUT_END_SEC가 원시 기록 범위(RAW_START_SEC~RAW_END_SEC) 밖입니다")
    return s0, s1


def speed_column(spd):
    s = np.round(spd, SPEED_DECIMALS)
    if isinstance(NO_MEASURE_SPEED, (int, float)) and not isinstance(NO_MEASURE_SPEED, bool):
        return np.nan_to_num(s, nan=float(NO_MEASURE_SPEED))
    return np.array([NO_MEASURE_SPEED if np.isnan(x) else str(float(x)) for x in s], dtype=object)


def volume_column(vol, spd):
    v = half_up(vol, 0).astype(int)
    if VOLUME_MIN_ONE_WITH_SPEED:
        v[(v == 0) & (np.nan_to_num(np.round(spd, SPEED_DECIMALS), nan=0.0) > 0)] = 1
    return v


def to_frame(det, sid, spd, vol, occ, runs):
    s0, s1 = output_bins(); nb = s1 - s0; nd = len(det)
    spd, vol, occ = spd[:, s0:s1].ravel(), vol[:, s0:s1].ravel(), occ[:, s0:s1].ravel()
    return pd.DataFrame({
        "scenario_id": sid,
        "time_sec": np.tile(RAW_START_SEC + BIN_SEC * np.arange(s0, s1), nd),
        "detector_location_id": np.repeat(det.detector_location_id.values, nb),
        "detector_id": np.repeat(det.detector_id.values, nb),
        "lane_id": np.repeat(det.lane_id.values, nb),
        "speed_kmh": speed_column(spd),
        "volume": volume_column(vol, spd),
        "occupancy_pct": half_up(occ, OCCUPANCY_DECIMALS),
        "aggregation_sec": BIN_SEC, "run_type": RUN_TYPE, "notes": NOTES.format(runs=runs)})[COLS]


def output_path(section, sid, runs, out_dir):
    return os.path.join(out_dir, section["out_dir"], OUTPUT_FILE_NAME.format(sid=sid, runs=runs, suffix=section["suffix"]))


def process_scenario(task):
    """시나리오 1개: 시드 N개 집계 → 평균 → CSV.  → (출력 경로, 상태, 행 수, 정지 시드-칸 수)"""
    section, set_type, sid, files, runs, det, out_dir, overwrite = task
    out = output_path(section, sid, runs, out_dir)
    try:
        if os.path.exists(out) and not overwrite:
            return out, "건너뜀(이미 있음)", 0, 0
        idx = detector_index(det)
        seeds = [seed_arrays(f, idx) for f in files]
        df = to_frame(det, sid, *average_seeds(seeds), runs)
        os.makedirs(os.path.dirname(out), exist_ok=True)
        part = out + ".partial"                         # 쓰는 중에 멈춰도 망가진 결과 파일이 남지 않게
        df.to_csv(part, index=False, encoding=OUTPUT_ENCODING)
        os.replace(part, out)
        s0, s1 = output_bins()
        return out, "완료", len(df), int(sum(s["stopped"][:, s0:s1].sum() for s in seeds))
    except Exception as e:
        return out, f"오류: {type(e).__name__}: {e}", 0, 0

## 4. 실행

아래 셀에서 처리할 범위를 고르고 실행합니다.

| 변수 | 뜻 | 예 |
|---|---|---|
| `RUN_SECTIONS` | 구간 (None = 전체 12개) | `["진입구간", "본선부-구간1", "영향권1(진출)"]` |
| `RUN_SETS` | 세트 (None = 추정용·검증용) | `["검증용"]` |
| `RUN_SCENARIOS` | 시나리오 번호 (None = 전체) | `[1, 49]` |
| `TEST_ONLY` | True면 첫 시나리오 1개만 (시험용) | |

- 전체(1,680개)는 joblib을 쓰면 3~4시간쯤 걸립니다. 스레드로 처리하면 더 오래 걸립니다.
- 중간에 멈췄다가 다시 실행하면 이미 만든 파일은 건너뜁니다.

In [ ]:
RUN_SECTIONS = None       # 예: ["진입구간"]
RUN_SETS = None           # 예: ["추정용"]
RUN_SCENARIOS = None      # 예: [1, 49]
TEST_ONLY = True          # 처음에는 True로 시험해 보고, 문제없으면 False로 바꿔 전체 실행


def run(sections=None, sets=None, scenarios=None, test_only=False, out_dir=None, workers=None, overwrite=None):
    out_dir = out_dir or OUTPUT_DIR; workers = workers or WORKERS
    overwrite = OVERWRITE if overwrite is None else overwrite
    if test_only:
        sections, sets = sections or [SECTIONS[0]["key"]], sets or [SETS[0]]
    print("=== 1단계: 원시 자료 확인 ===")
    tasks, det, problems = build_tasks(sections, sets, set(scenarios) if scenarios else None)
    if test_only:
        tasks = tasks[:1]
    if not tasks:
        print("처리할 작업이 없습니다."); return None
    jobs = [(sec, st, sid, files, n, det, out_dir, overwrite) for sec, st, sid, files, n in tasks]
    W = max(1, min(workers, len(jobs)))
    print(f"\n=== 2·3단계: 시드별 집계 → {tasks[0][4]}회 평균 → CSV ===")
    print(f"결과 폴더: {out_dir} | 동시 처리 {W} ({'joblib 프로세스' if HAS_JOBLIB else '스레드'}) | 시나리오 {len(jobs)}개")
    t0 = time.time(); log = []

    def results():
        if HAS_JOBLIB:
            try:
                yield from Parallel(n_jobs=W, backend="loky", return_as="generator_unordered")(delayed(process_scenario)(j) for j in jobs)
            except TypeError:                           # 오래된 joblib
                yield from Parallel(n_jobs=W, backend="loky", return_as="generator")(delayed(process_scenario)(j) for j in jobs)
        else:
            with ThreadPoolExecutor(W) as ex:
                for fu in as_completed([ex.submit(process_scenario, j) for j in jobs]):
                    yield fu.result()

    key = {output_path(sec, sid, n, out_dir): (sec["key"], st, sid) for sec, st, sid, files, n, *_ in jobs}
    for k, (out, status, rows, n_stop) in enumerate(results(), 1):
        sk, st, sid = key[out]
        log.append(dict(구간=sk, 세트=st, scenario_id=sid, 상태=status, 행수=rows, 정지_시드칸=n_stop, 파일=os.path.relpath(out, out_dir)))
        if status.startswith("오류"):
            print(f"\n  오류 — {sk} 시나리오 {sid}: {status}")
        p = k / len(jobs); el = time.time() - t0; eta = el / k * (len(jobs) - k)
        print(f"\r[{'#' * int(p * 30):<30}] {k}/{len(jobs)} {p:.0%}  경과 {el / 60:.1f}분  남은 시간 약 {eta / 60:.0f}분", end="", flush=True)
    d = pd.DataFrame(log).sort_values(["구간", "scenario_id"]).reset_index(drop=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    os.makedirs(out_dir, exist_ok=True)
    d.to_csv(os.path.join(out_dir, f"실행기록_{stamp}.csv"), index=False, encoding="utf-8-sig")
    print(f"\n\n끝 ({(time.time() - t0) / 60:.1f}분) — " + ", ".join(f"{a} {b}개" for a, b in d.상태.str.split(":").str[0].value_counts().items()))
    print(f"실행 기록: {os.path.join(out_dir, f'실행기록_{stamp}.csv')}")
    return d


LOG = run(RUN_SECTIONS, RUN_SETS, RUN_SCENARIOS, TEST_ONLY)

## 5. 결과 확인
방금 실행한 기록을 요약하고, 결과 파일 하나를 미리 봅니다. 속도 컬럼이 숫자 / 0.0 / NA 중 무엇인지 비율로 보여 줍니다.

In [ ]:
if LOG is not None and len(LOG):
    print(LOG.groupby(["구간", "상태"]).size().rename("시나리오 수").to_string())
    done = LOG[LOG.상태 == "완료"]
    if len(done):
        f = os.path.join(OUTPUT_DIR, done.파일.iloc[0])
        x = pd.read_csv(f, dtype={"speed_kmh": str}, keep_default_na=False)
        sp = x.speed_kmh
        kind = np.where(sp == "NA", "NA (측정 없음)", np.where(sp == "0.0", "0.0 (정체)", "숫자 (주행)"))
        print(f"\n파일: {done.파일.iloc[0]}  ({len(x):,}행)")
        print(pd.Series(kind).value_counts().rename("행 수").to_string())
        display(x.head(10)) if "display" in globals() else print(x.head(10).to_string())